In [0]:
import requests
import time
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
target_table_name = "finance_intelligence_hub.bronze.company_news_polygon"

# Only used the very first time the table doesn't exist yet.
# After the table exists, the script figures out where to resume on its own.
default_start_date_str = "2026-07-03"
end_date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Re-pull this many days before the last loaded date, to catch late-arriving
# or backfilled articles. Safe because MERGE dedupes on source_url.
lookback_days = 1

# Retry caps so a bad key / outage can't hang the job forever
max_day_retries = 5
max_rate_limit_retries = 10

api_key = "9dsWtWB7r0h5ZXqzfxVcdo1VoGmHSF1o"

# ---------------------------------------------------------------------------
# Determine incremental start date (this is what makes the load incremental)
# ---------------------------------------------------------------------------
table_exists = spark.catalog.tableExists(target_table_name)

if table_exists:
    last_loaded_row = spark.sql(
        f"SELECT MAX(published_date) AS max_date FROM {target_table_name}"
    ).collect()[0]
    last_loaded = last_loaded_row["max_date"]

    if last_loaded:
        last_loaded_date = datetime.strptime(last_loaded[:10], "%Y-%m-%d").date()
        current_date = last_loaded_date - timedelta(days=lookback_days)
    else:
        current_date = datetime.strptime(default_start_date_str, "%Y-%m-%d").date()
else:
    current_date = datetime.strptime(default_start_date_str, "%Y-%m-%d").date()

end_date = datetime.strptime(end_date_str, "%Y-%m-%d").date()

print("=" * 70)
print(f"Table exists: {table_exists}")
print(f"Incremental load window: {current_date} -> {end_date}")
print("=" * 70)

grand_total_saved = 0
table_created_this_run = False

# ---------------------------------------------------------------------------
# Main loop: process one day at a time
# ---------------------------------------------------------------------------
while current_date <= end_date:

    date_str = current_date.strftime("%Y-%m-%d")
    print(f"\nProcessing date: {date_str}")

    url = "https://api.polygon.io/v2/reference/news"
    params = {
        "published_utc.gte": f"{date_str}T00:00:00Z",
        "published_utc.lte": f"{date_str}T23:59:59Z",
        "order": "desc",
        "limit": 1000,
        "apiKey": api_key
    }

    day_news_count = 0
    page = 1
    all_records = []
    day_retry_count = 0
    rate_limit_retry_count = 0

    while url:

        response = requests.get(url, params=params if "apiKey" in params else None)

        if response.status_code == 200:
            day_retry_count = 0
            rate_limit_retry_count = 0

            data = response.json()
            results = data.get("results", [])

            if not results:
                break

            run_loaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

            for news in results:
                title = news.get("title", "")
                description = news.get("description", "")
                published_date = news.get("published_utc", "")
                full_text = f"{title}. {description}".strip()

                tickers = news.get("tickers", [])
                related_tickers = ",".join(tickers) if tickers else "GENERAL"
                primary_ticker = tickers[0] if tickers else "GENERAL"

                all_records.append({
                    "ticker": primary_ticker,
                    "published_date": published_date,
                    "title": title,
                    "description": description,
                    "text_for_finbert": full_text,
                    "author": news.get("author", ""),
                    "source_url": news.get("article_url", ""),
                    "image_url": news.get("image_url", ""),
                    "publisher_name": news.get("publisher", {}).get("name", ""),
                    "publisher_url": news.get("publisher", {}).get("homepage_url", ""),
                    "related_tickers": related_tickers,
                    "dwh_loaded_at": run_loaded_at,
                })

            day_news_count += len(results)
            print(f"   Page {page}: {len(results)} articles retrieved (Daily total: {day_news_count})")
            page += 1

            next_url = data.get("next_url")
            if next_url:
                url = f"{next_url}&apiKey={api_key}"
                params = {}
                time.sleep(12)
            else:
                url = None

        elif response.status_code == 429:
            rate_limit_retry_count += 1
            if rate_limit_retry_count > max_rate_limit_retries:
                raise RuntimeError(
                    f"Rate limited on {date_str} after {max_rate_limit_retries} retries. Aborting."
                )
            print(f"Rate limit reached ({rate_limit_retry_count}/{max_rate_limit_retries}). Waiting 30s...")
            time.sleep(30)
            continue

        else:
            day_retry_count += 1
            if day_retry_count > max_day_retries:
                raise RuntimeError(
                    f"Failed for {date_str} after {max_day_retries} retries. "
                    f"Status={response.status_code}, body={response.text[:200]}"
                )
            print(
                f"Failed to retrieve news for {date_str}. Status: {response.status_code}. "
                f"Retry {day_retry_count}/{max_day_retries} in 15s..."
            )
            time.sleep(15)
            continue

    # -----------------------------------------------------------------
    # Save this day's data — MERGE, never overwrite, once table exists
    # -----------------------------------------------------------------
    if all_records:

        spark_df = spark.createDataFrame(all_records)
        # Bronze pattern: all-STRING schema
        for col_name in spark_df.columns:
            spark_df = spark_df.withColumn(col_name, F.col(col_name).cast("string"))

        if not spark.catalog.tableExists(target_table_name):
            # Only path that ever creates/overwrites — and only because the
            # table genuinely does not exist yet. Never runs again after this.
            spark_df.write.format("delta").mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(target_table_name)
            table_created_this_run = True
            print(f"Table {target_table_name} created.")
        else:
            spark_df.createOrReplaceTempView("staging_news")
            spark.sql(f"""
                MERGE INTO {target_table_name} AS target
                USING staging_news AS source
                ON target.source_url = source.source_url
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)

        grand_total_saved += len(all_records)
        all_records.clear()

    print(f"Finished processing {date_str} successfully.")
    current_date += timedelta(days=1)

print("\n" + "=" * 70)
print("Incremental extraction completed successfully.")
print(f"Total articles processed this run: {grand_total_saved}")
print("=" * 70)